# Mitochondrial fragmentation index

A continuous, calibrated score for how fragmented the mitochondria in a confocal
image are, learned from three blind observers' majority call rather than from a
threshold anyone chose.

This notebook covers the two halves of the pipeline that are portable Python:

| | What it does | Section |
|---|---|---|
| **Training** | features to model to calibrated index, validated on unseen imaging sessions | 1-6 |
| **Usage** | segmentation mask in, index out, with a quality gate | 7 |

**What is not here.** Ingest and segmentation run in Fiji with the Labkit plugin,
which is a Java GUI and does not run in Colab. Those steps produce the masks this
notebook consumes. See `PIPELINE.md` in the repository.

Runtime: about two minutes, except the learning curve in section 6, which is
marked and optional.

---

## 0. Setup

In [ ]:
# Colab already has numpy, pandas, scipy, scikit-learn, matplotlib and tifffile.
# scikit-image is the only one usually missing.
import importlib, subprocess, sys
for mod, pkg in [('skimage', 'scikit-image'), ('tifffile', 'tifffile')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
from scipy import stats
print('numpy', np.__version__, '| pandas', pd.__version__)

### Get the data

Three small CSVs drive everything in sections 1-6. If the repository is public,
the next cell fetches them. Otherwise set `SOURCE = 'upload'` and upload
`features_v5.csv`, `labels_consensus.csv` and `covariates.csv` when prompted.

In [ ]:
SOURCE = 'github'          # 'github' | 'upload' | 'local'
REPO = 'https://raw.githubusercontent.com/vishal9045/mitofi/master'
FILES = ['features_v5.csv', 'labels_consensus.csv', 'covariates.csv']

import os, urllib.request, urllib.error
os.makedirs('data', exist_ok=True)

if SOURCE == 'github':
    try:
        for f in FILES:
            urllib.request.urlretrieve(f'{REPO}/data/{f}', f'data/{f}')
        print('fetched from GitHub')
    except urllib.error.HTTPError as e:
        raise SystemExit(f'{e} - the repository may still be private. Set SOURCE = "upload" and re-run this cell.')
elif SOURCE == 'upload':
    from google.colab import files
    for name, blob in files.upload().items():
        open(f'data/{name}', 'wb').write(blob)

feat   = pd.read_csv('data/features_v5.csv')
labels = pd.read_csv('data/labels_consensus.csv')
cov    = pd.read_csv('data/covariates.csv')
print(feat.shape, labels.shape, cov.shape)

### Chart style

One palette, defined once. The three calls always take the same three hues, in the
same order, so a colour means the same thing in every figure below.

In [ ]:
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#8a8983", "grid": "#e6e5e1", "surface": "#fcfcfb"}
CALLS = ['fragmented', 'normal', 'elongated']
CALL_C = {'fragmented': C['blue'], 'normal': C['orange'], 'elongated': C['aqua']}

mpl.rcParams.update({
    'figure.facecolor': C['surface'], 'axes.facecolor': C['surface'],
    'savefig.facecolor': C['surface'],
    'axes.edgecolor': C['grid'], 'axes.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelcolor': C['ink2'], 'axes.titlecolor': C['ink'],
    'axes.titlesize': 12, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'axes.titlepad': 12, 'axes.labelsize': 10,
    'xtick.color': C['ink2'], 'ytick.color': C['ink2'],
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'grid.color': C['grid'], 'grid.linewidth': 0.8,
    'legend.frameon': False, 'legend.fontsize': 9,
    'font.size': 10, 'lines.linewidth': 2, 'figure.dpi': 120,
})

def tidy(ax, ygrid=True):
    """Recessive grid behind the marks, never over them."""
    ax.set_axisbelow(True)
    ax.grid(axis='y' if ygrid else 'x', alpha=0.9)
    ax.grid(axis='x' if ygrid else 'y', visible=False)
    return ax

---
## 1. What the model sees

One row per image. 43 shape measurements from the segmentation mask, in physical
units from that image's own pixel size, plus the call two of three blind raters
made. Images any rater could not read are `uncertain`; three-way disagreements
are `split`. Neither is trained on.

No pixels reach the model. It sees this table and nothing else.

In [ ]:
d = feat.merge(labels[['image_id', 'blinded_call']], on='image_id')

print(f'{len(d)} images, {len(set(d.session))} imaging sessions')
print(f'pixel size spans {d.px_um.min():.4f} to {d.px_um.max():.4f} um ({d.px_um.max()/d.px_um.min():.1f}-fold)')
print()
print(d.blinded_call.value_counts().to_string())

d[['image_id', 'blinded_call', 'n_objects', 'p90_aspect',
   'areawt_formfactor', 'hole_area_frac']].head()

---
## 2. Which measurements carry the signal

Two questions per feature, and the second is the one that decides the design:

1. Does it separate the three calls? Effect size is epsilon-squared from a
   Kruskal-Wallis test.
2. Does it also track *how the image was acquired* - pixel size or brightness?
   A feature that does is measuring the microscope, not the biology.

The pool spans a 4.3-fold range of pixel sizes, so this is not hypothetical.

In [ ]:
SKIP = {'image_id', 'session', 'blinded_call', 'frame_px', 'sizeZ'}
m3 = d[d.blinded_call.isin(CALLS)]

rows = []
for c in [c for c in d.columns if c not in SKIP and d[c].dtype != object]:
    groups = [m3.loc[m3.blinded_call == k, c].values for k in CALLS]
    H, p = stats.kruskal(*groups)
    rows.append(dict(feature=c, eps2=(H - 2) / (len(m3) - 3), p=p,
                     r_px=abs(stats.spearmanr(m3[c], m3.px_um).statistic),
                     r_int=abs(stats.spearmanr(m3[c], m3.img_mean_int).statistic)))
rank = pd.DataFrame(rows).sort_values('eps2', ascending=False).reset_index(drop=True)
rank.head(8).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.6))
r = rank[rank.eps2 > 0.01]
clean = (r.r_px <= 0.4) & (r.r_int <= 0.4)      # both screens, as CLEAN below

ax.scatter(r.loc[clean, 'r_px'], r.loc[clean, 'eps2'], s=64,
           color=C['aqua'], edgecolor=C['surface'], linewidth=2,
           label='clean on both screens - usable', zorder=3)
ax.scatter(r.loc[~clean, 'r_px'], r.loc[~clean, 'eps2'], s=64,
           color=C['orange'], edgecolor=C['surface'], linewidth=2,
           label='tracks acquisition - excluded', zorder=3)
ax.axvline(0.4, color=C['muted'], linewidth=1, linestyle='--', zorder=1)
ax.text(0.405, ax.get_ylim()[1] * 0.97, ' exclusion threshold', fontsize=8.5,
        color=C['muted'], va='top')

for _, row in r.nlargest(4, 'eps2').iterrows():
    ax.annotate(row.feature, (row.r_px, row.eps2), fontsize=8.5, color=C['ink2'],
                xytext=(8, -1), textcoords='offset points', va='center')

ax.set_xlabel('|Spearman rho| with pixel size')
# the vertical rule marks the pixel-size screen; the intensity screen is not on this axis
ax.set_ylabel('effect size separating the calls')
ax.set_title('The features that separate best are the ones\nindependent of magnification')
ax.legend(loc='upper right')
tidy(ax)
plt.tight_layout(); plt.show()

Everything in orange is expressed in micrometres or square micrometres. The
conversions are correct - the extraction code is tested against synthetic objects
measured at two pixel sizes - so the correlation is a property of the data, not a
units bug. Coarser sampling merges thin structures and loses small objects, and in
this pool pixel size also co-varies with cell type.

Either way the conclusion holds: **build the index from the dimensionless
descriptors.**

---
## 3. Fitting the model

Eight features survive both screens. The model is multinomial logistic regression:
eight coefficients per class, twenty-four numbers in total, all readable.

**Validation holds out whole imaging sessions.** A random split would put images
from the same dish in training and test, and the model would score well by
recognising that dish's illumination. Both are run below so the difference is
visible rather than asserted.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix

CLEAN = ['p90_aspect', 'p90_ecc', 'areawt_formfactor', 'med_aspect', 'med_ecc',
         'hole_area_frac', 'areawt_aspect', 'frac_fg_largest']
# The shipped model reads a ninth feature that does NOT pass the screen above:
# mean_skel_len_per_obj_um correlates 0.406 with image mean intensity against a
# cutoff of 0.400. It is kept because the screen tests the marginal correlation
# of a FEATURE with a nuisance variable, while what matters is the correlation
# of the INDEX -- adding it moves the index's pixel-size dependence from rho
# -0.09 to -0.00. It buys no significant accuracy (+0.017, 95% CI -0.002 to
# +0.040). See scripts/stage7_index.py.
MODEL_FEATURES = CLEAN + ['mean_skel_len_per_obj_um']
SEED, FOLDS = 20260905, 5

m = m3.reset_index(drop=True)
y, g = m.blinded_call.values, m.session.values
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))

by_session = list(GroupKFold(n_splits=FOLDS).split(m, y, g))
at_random  = list(StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED).split(m, y))

res = {}
res['majority baseline'] = cross_val_predict(DummyClassifier(strategy='most_frequent'),
                                             m[MODEL_FEATURES], y, cv=by_session)
res['logistic, by session'] = cross_val_predict(lr(), m[MODEL_FEATURES], y, cv=by_session)
res['random forest, by session'] = cross_val_predict(
    RandomForestClassifier(n_estimators=500, min_samples_leaf=3, random_state=SEED, n_jobs=-1),
    m[MODEL_FEATURES], y, cv=by_session)
res['logistic, random split'] = cross_val_predict(lr(), m[MODEL_FEATURES], y, cv=at_random)

for k, v in res.items():
    print(f'{k:<28} accuracy {accuracy_score(y, v):.3f}   balanced {balanced_accuracy_score(y, v):.3f}')

The gap between held-out-by-session and the random split is about one point.

That is worth pausing on: it means the model is not succeeding by memorising
acquisition conditions. A pool spanning 36 sessions, several cell types and a
four-fold magnification range does not leave it that option.

In [ ]:
pred = res['logistic, by session']
cm = confusion_matrix(y, pred, labels=CALLS)
cmn = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5.0, 4.2))
ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=11,
                color='white' if cmn[i, j] > 0.55 else C['ink'])
ax.set_xticks(range(3), CALLS, rotation=20, ha='right')
ax.set_yticks(range(3), CALLS)
ax.set_xlabel('predicted'); ax.set_ylabel('majority said')
ax.set_title('Errors sit in the middle class')
ax.set_xticks(np.arange(-.5, 3), minor=True)
ax.set_yticks(np.arange(-.5, 3), minor=True)
ax.grid(which='minor', color=C['surface'], linewidth=2)
ax.grid(which='major', visible=False)
ax.tick_params(which='minor', length=0)
for s in ax.spines.values():
    s.set_visible(False)
plt.tight_layout(); plt.show()

for i, c in enumerate(CALLS):
    print(f'{c:<12} {cm[i, i]:>3} of {cm[i].sum():>3} recovered')

The two ends are recovered well; the middle is not. That is what a middle category
on an ordered scale does when it is forced to behave like a discrete class, and it
is the direct argument for reporting a number instead of a label.

---
## 4. From probabilities to an index

Collapse the three class probabilities into one number:

$$\text{index} = 1.0 \cdot P(\text{fragmented}) + 0.5 \cdot P(\text{normal}) + 0 \cdot P(\text{elongated})$$

1 is fully fragmented, 0 is a fully elongated network.

In [ ]:
prob = cross_val_predict(lr(), m[MODEL_FEATURES], y, cv=by_session, method='predict_proba')
cls = list(np.unique(y))
index = prob[:, cls.index('normal')] * 0.5 + prob[:, cls.index('fragmented')] * 1.0
m = m.assign(index=index)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
rng = np.random.default_rng(0)
for k, call in enumerate(CALLS):
    v = m.loc[m.blinded_call == call, 'index'].values
    ax.scatter(v, k + rng.uniform(-0.16, 0.16, len(v)), s=26, alpha=0.55,
               color=CALL_C[call], edgecolor='none', zorder=3)
    ax.plot([np.median(v)] * 2, [k - 0.30, k + 0.30], color=C['ink'],
            linewidth=2.5, zorder=4)
    ax.text(np.median(v), k + 0.42, f'median {np.median(v):.2f}', fontsize=8.5,
            color=C['ink2'], ha='center')

ax.set_yticks(range(3), CALLS)
ax.set_xlabel('fragmentation index    (0 = elongated,  1 = fragmented)')
ax.set_xlim(-0.02, 1.02)
ax.set_title('Index against the raters\' call, out of fold')
tidy(ax, ygrid=False)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score
ends = np.isin(y, ['fragmented', 'elongated'])
auc = roc_auc_score((y[ends] == 'fragmented').astype(int), index[ends])
print(f'fragmented vs elongated, out of fold: AUC {auc:.3f}  (n {ends.sum()})')

---
## 5. Is the confidence honest?

The model reports a probability alongside every call. Calibration asks whether
that number means anything: when it says 80%, is it right 80% of the time?

In [ ]:
top = prob.max(axis=1)
hit = np.array(cls)[prob.argmax(axis=1)] == y
bins = [(.33, .5), (.5, .6), (.6, .7), (.7, .8), (.8, 1.01)]
pts = [(top[s].mean(), hit[s].mean(), s.sum())
       for s in ((top >= a) & (top < b) for a, b in bins) if s.sum()]
px_, py_, pn = zip(*pts)

fig, ax = plt.subplots(figsize=(4.8, 4.4))
ax.plot([0.3, 1], [0.3, 1], color=C['muted'], linewidth=1, linestyle='--', zorder=1)
ax.text(0.94, 0.90, 'perfect', fontsize=8.5, color=C['muted'], ha='right', rotation=38)
ax.plot(px_, py_, color=C['blue'], marker='o', markersize=8,
        markeredgecolor=C['surface'], markeredgewidth=2, zorder=3)
for x, yv, n in pts:
    ax.annotate(f'n={n}', (x, yv), xytext=(6, -11), textcoords='offset points',
                fontsize=8, color=C['muted'])
ax.set_xlabel('probability the model reported'); ax.set_ylabel('how often it was right')
ax.set_title('Reported confidence is close to honest')
ax.set_xlim(0.3, 1); ax.set_ylim(0.3, 1)
tidy(ax)
plt.tight_layout(); plt.show()

### The mixed band

The ends of the scale separate cleanly and the middle does not, so the index
declines to commit in the middle. The rule was fixed before the result was looked
at: the cuts are placed so that at least 90% of images beyond each one carry the
corresponding call.

The purity requirement is a choice, so its consequences are reported rather than
a single flattering number.

In [ ]:
def band_edges(idx, y, purity):
    lo = hi = None
    for t in np.unique(np.round(idx, 3)):
        s = y[idx < t]
        if len(s) >= 20 and (s == 'elongated').mean() >= purity: lo = t
    for t in np.unique(np.round(idx, 3))[::-1]:
        s = y[idx > t]
        if len(s) >= 20 and (s == 'fragmented').mean() >= purity: hi = t
    return lo, hi

for p in (0.80, 0.85, 0.90, 0.95):
    a, b = band_edges(index, y, p)
    if a is None or b is None:
        print(f'{p:.0%}  no band satisfies this'); continue
    print(f'{p:.0%}  band {a:.3f} to {b:.3f}   {((index < a) | (index > b)).mean():.1%} of images get a call')

LO, HI = band_edges(index, y, 0.90)

An 80% abstention rate looks alarming and is mostly a property of the rule. The
rule counts a `normal` image above the upper cut as an impurity, although an index
that places a mixed field high on a fragmentation scale is not making an error.
What the band really measures is how cleanly the *extremes* isolate: with high
purity, and low coverage.

---
## 6. Would more data help? *(optional, ~2 minutes)*

Before anyone rates another 500 images or reaches for a deeper model, it is worth
measuring whether the current one is still learning.

In [ ]:
REPEATS = 5
sizes = [40, 80, 120, 180, 240, 300, min(len(tr) for tr, _ in by_session)]
curve = []
for n in sizes:
    sc = []
    for rep in range(REPEATS):
        rg = np.random.default_rng(SEED + rep)
        for tr, te in by_session:
            if n > len(tr): continue
            pick = rg.choice(tr, n, replace=False)
            if len(np.unique(y[pick])) < 3: continue
            mdl = lr().fit(m[MODEL_FEATURES].iloc[pick], y[pick])
            sc.append(balanced_accuracy_score(y[te], mdl.predict(m[MODEL_FEATURES].iloc[te])))
    curve.append((n, np.mean(sc), np.std(sc)))

cn, cm_, cs = map(np.array, zip(*curve))
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.fill_between(cn, cm_ - cs, cm_ + cs, color=C['blue'], alpha=0.13, zorder=2)
ax.plot(cn, cm_, color=C['blue'], marker='o', markersize=8,
        markeredgecolor=C['surface'], markeredgewidth=2, zorder=3)
ax.set_ylim(0.50, 0.70)   # chance is 0.333; a tighter axis would exaggerate a 4-point rise
ax.set_xlabel('training images'); ax.set_ylabel('balanced accuracy, held out by session')
ax.set_title('The curve flattens well before the data runs out')
tidy(ax)
plt.tight_layout(); plt.show()

print(f'{cm_[0]:.3f} at {cn[0]} images  ->  {cm_[-1]:.3f} at {cn[-1]}, with a fold-to-fold sd of about {cs.mean():.3f}')

Flat past roughly 240 images, with the remaining differences smaller than the
fold-to-fold spread. Rating several hundred more would add nothing measurable, and
a higher-capacity model would start from a worse position on the one axis this
actually measures.

The limits lie elsewhere: how far the raters agree with each other, and the segmentation
itself.

---
## 7. Usage: scoring new images

Everything above builds the index. This section applies it.

Input is a **segmentation mask** - a binary TIFF from the Labkit classifier - plus
the pixel size in micrometres. Pixel size cannot be guessed and must be supplied;
the index is computed in physical units and is meaningless without it.

### The quality gate

The index cannot tell a genuinely fragmented cell from an image too poor to read.
Both produce many small disconnected objects, and the model scores the second one
confidently. So a gate sits in front of it, with thresholds derived from the 35
images rater 1 marked unreadable:

| Rule | Meaning |
|---|---|
| `fg_frac < 0.01` | segmentation collapsed |
| `p90_major_um < 2.25` | the mask is speckle - even the largest objects are too small to be mitochondria |

On the pilot pool that catches 30 of the 35 unreadable images, at the cost of
rejecting 39 of the 465 readable ones.

**A rejected image gets no index.**

In [ ]:
# 24 example masks, including a few the gate should reject
import urllib.request, zipfile, io, glob
if not glob.glob('data/example_masks/*.tif'):
    try:
        urllib.request.urlretrieve(f'{REPO}/data/example_masks.zip', 'ex.zip')
        zipfile.ZipFile('ex.zip').extractall('data/')
    except Exception as e:
        print('could not fetch examples:', e)
        print('upload your own masks to data/example_masks/ instead')
print(len(glob.glob('data/example_masks/*.tif')), 'masks')

In [ ]:
import math, tifffile
from scipy import ndimage as ndi
from skimage.measure import label, regionprops

MIN_OBJ_UM2, MIN_FG_FRAC, MIN_P90_MAJOR_UM = 0.05, 0.01, 2.25

def shape_features(mask, px):
    """The subset of the 43 features the index and the gate need."""
    apx = px * px
    props = [p for p in regionprops(label(mask, connectivity=2)) if p.area * apx >= MIN_OBJ_UM2]
    f = {'fg_frac': mask.sum() / mask.size, 'n_objects': len(props)}
    if not props:
        return f
    A   = np.array([p.area * apx for p in props])
    P   = np.array([p.perimeter * px for p in props])
    maj = np.array([p.axis_major_length * px for p in props])
    mnr = np.array([p.axis_minor_length * px for p in props])
    ar  = maj / np.maximum(mnr, px)
    ff  = 4 * math.pi * A / np.maximum(P, px) ** 2
    ecc = np.array([p.eccentricity for p in props])
    hole = ndi.binary_fill_holes(mask) & ~mask
    f.update(p90_aspect=np.percentile(ar, 90), med_aspect=np.median(ar),
             p90_ecc=np.percentile(ecc, 90), med_ecc=np.median(ecc),
             p90_major_um=np.percentile(maj, 90),
             areawt_formfactor=np.average(ff, weights=A),
             areawt_aspect=np.average(ar, weights=A),
             hole_area_frac=hole.sum() / max(mask.sum(), 1),
             frac_fg_largest=A.max() / A.sum())
    return f

def qc(f):
    if f.get('n_objects', 0) == 0:            return False, 'no objects in mask'
    if f['fg_frac'] < MIN_FG_FRAC:            return False, f"collapsed: foreground {f['fg_frac']:.2%}"
    if f['p90_major_um'] < MIN_P90_MAJOR_UM:  return False, f"speckle: p90 major {f['p90_major_um']:.2f} um"
    return True, ''

In [ ]:
final = lr().fit(m[MODEL_FEATURES], y)          # fitted on everything, for scoring new data
px_map = pd.read_csv('data/example_masks/px_um.csv').set_index('image_id').px_um.to_dict()

out = []
for path in sorted(glob.glob('data/example_masks/*.tif')):
    iid = os.path.basename(path).replace('.ome_segmentation.tif', '')
    f = shape_features(tifffile.imread(path).astype(bool), px_map[iid])
    ok, why = qc(f)
    row = dict(image_id=iid, qc_pass=ok, qc_reason=why)
    if all(k in f for k in MODEL_FEATURES):
        p = final.predict_proba(np.array([[f[k] for k in MODEL_FEATURES]]))[0]
        i = p[cls.index('normal')] * 0.5 + p[cls.index('fragmented')] * 1.0
        row.update(index=round(i, 3) if ok else np.nan,
                   band='fragmented' if i > HI else 'elongated' if i < LO else 'mixed',
                   confidence=round(p.max(), 2))
    out.append(row)

scores = pd.DataFrame(out)
print(f"{scores.qc_pass.sum()} of {len(scores)} passed the gate")
scores

### Use your own masks

Upload binary TIFF masks, set the pixel size, and re-run the cell above with
`data/example_masks` pointed at your folder.

```python
from google.colab import files
files.upload()      # then move the files into a folder and set px_um
```

Two things that must match, or the numbers are not comparable to anything:

- **The same segmentation classifier.** Two classifiers trained by the same person
  on the same images, differing only in where the scribbles went, agreed on 67% of
  segmented pixels. Masks from a different classifier produce a different index.
- **The same pixel-size convention**, in micrometres per pixel.

The gate thresholds, the band cuts and the classifier were all derived on one
microscope. Applying them to another instrument without revalidating is not
supported by anything measured here.

---
## What this index is, and is not

**It is** an instrument for measuring mitochondrial morphology, calibrated against
three blind observers, reproducible on imaging sessions it has never seen, and free of
any threshold a user could tune toward a preferred answer.

**It is not** an assay for a treatment. It was never trained on, or validated
against, any experimental condition.

Known limits, all measured rather than assumed:

- Residual correlation of about -0.21 with mean image intensity. For an instrument
  meant to read morphology regardless of acquisition, this is the most important
  number still to reduce.
- Accuracy is bounded by how far people agree: three blind raters agree pairwise
  on 67-69% of images. The model has seen all three, so no unseen rater is left
  to test it against.
- Features cover the whole field while the raters judged the dominant cell, so on
  multi-cell fields the label and the measurement describe different subjects.

Full method record and code: `github.com/vishal9045/mitofi`